In [18]:
from bs4 import BeautifulSoup
from src.utils.gembase_utils import GembaseUtils
import json

from src.db.mysql import DbConnection

conn = DbConnection()

rows_apps_ids = conn.select_all("""
select app_id_int
  from scraped_data.apps
 WHERE app_id_int not in (select app_id_int from app_temp_data.erik_app_store_tags)
""")

cnt = 0
for row in rows_apps_ids:

    app_id_int = row["app_id_int"]
    row_data = conn.select_one_or_none("""
    select a.store, d.data
    from scraped_data.apps_source_data d,
         scraped_data.apps a
    where a.app_id_int = d.app_id_int
    and a.app_id_int = %s
    and ((a.store = 1 and d.data_type = 'scrap_data')
    or (a.store = 2 and d.data_type = 'store_page'))
    """, [app_id_int]
  )

    if row_data is None:
        continue

    cnt += 1

    bulk_data = []
    if row_data["store"] == 1:
        data = json.loads(GembaseUtils.decompress(row_data["data"]))
        if 'genres' in data:
            for tag in data["genres"]:
                bulk_data.append((app_id_int, tag, 1))
        if "tags" in data:
            for tag in data["tags"]:
                bulk_data.append((app_id_int, tag, 0))
    else:
        data = GembaseUtils.decompress(row_data["data"])
        bs = BeautifulSoup(data, "html.parser")
        app_tags = bs.find_all(name="a", attrs={"class": "app_tag"})
        if app_tags is not None:
            for it in app_tags:
                bulk_data.append((app_id_int, it.getText().strip(), 0))

    conn.bulk("""
    INSERT INTO app_temp_data.erik_app_store_tags (app_id_int, tag, genre)
    VALUES (%s, %s, %s)
    """, bulk_data)
    conn.commit()

    if cnt % 100 == 0:
        print(cnt)

conn.close()

In [8]:
from bs4 import BeautifulSoup
from src.utils.gembase_utils import GembaseUtils
import json
from src.runtime_constants import rr
from src.db.mysql import DbConnection

rr.set_env(rr.ENV_PROD, True)
conn = DbConnection()

conn.query("""
INSERT INTO scraped_data.def_store_tags (store, store_tag, store_tag_raw)
SELECT DISTINCT a.store, CONCAT('steam__', LOWER(t.tag)) as store_tag, t.tag as store_tag_raw
  FROM app_temp_data.erik_app_store_tags t,
       scraped_data.apps a
 WHERE a.app_id_int = t.app_id_int
   and a.store = 2
""")
conn.commit()
conn.query("""
INSERT INTO scraped_data.apps_store_tags (app_id_int, store_tag_id)
SELECT t.app_id_int, st.id
  FROM app_temp_data.erik_app_store_tags t,
       scraped_data.def_store_tags st,
       scraped_data.apps a
 WHERE st.store_tag = CONCAT('steam__', LOWER(t.tag))
 AND a.app_id_int = t.app_id_int
 AND a.store = 2
""")
conn.commit()

conn.close()

In [7]:
conn.close()

In [2]:
from src.runtime_constants import rr
from src import external_api
from src.db.mysql import DbConnection

rr.set_env(rr.ENV_PROD, True)
conn = DbConnection()

rows = conn.select_all("""
SELECT z1.tag, z1.store, GROUP_CONCAT(aa.title) AS top_5_apps, z1.total_num AS cnt FROM (
select t.tag, a.app_id_int, a.store,
 row_number() over (partition by t.tag order by a.installs DESC) as row_num,
 COUNT(1)  over (partition by t.tag) as total_num
  from app_temp_data.erik_app_store_tags t,
       scraped_data.apps a
 WHERE t.app_id_int = a.app_id_int
) z1,
scraped_data.apps aa
WHERE z1.row_num <= 5 AND aa.app_id_int = z1.app_id_int
GROUP BY z1.tag, z1.store, z1.total_num
""")

values_google = [["tag", "count", "games"]]
values_steam = [["tag", "count", "games"]]
for row in rows:
    if row["store"] == 1:
        values_google.append((row["tag"], row["cnt"], row["top_5_apps"]))
    else:
        values_steam.append((row["tag"], row["cnt"], row["top_5_apps"]))

conn.close()

external_api.write_sheet("1TIu0NXPH8LFgVmlzA4-y2GdI80MbnQW_-FAvAxDCvys", "store tags - google", values_google)
external_api.write_sheet("1TIu0NXPH8LFgVmlzA4-y2GdI80MbnQW_-FAvAxDCvys", "store tags - steam", values_steam)


{'spreadsheetId': '1TIu0NXPH8LFgVmlzA4-y2GdI80MbnQW_-FAvAxDCvys',
 'updatedRange': "'store tags - steam'!A1:C446",
 'updatedRows': 446,
 'updatedColumns': 3,
 'updatedCells': 1338}

In [19]:
from PIL import Image
from io import BytesIO
import requests
from src.db.mysql import DbConnection

conn = DbConnection()
rows = conn.select_all("""
SELECT app_id_int, icon FROM scraped_data.apps
WHERE app_id_int NOT IN (
SELECT app_id_int FROM scraped_data.apps_icons
)
and icon is not null
and app_id_int = 146
""")

cnt = 0
size = 128, 128
for row in rows:
    app_id_int = row["app_id_int"]
    icon = row["icon"]

    try:
        response = requests.get(icon)
        img = Image.open(BytesIO(response.content))
        img.thumbnail(size, Image.LANCZOS)
        output = BytesIO()
        img.save(output, format='PNG')
        data = output.getvalue()
        conn.query("delete from scraped_data.apps_icons where app_id_int = %s", [app_id_int])
        conn.query("insert into scraped_data.apps_icons (app_id_int, icon) VALUES (%s, %s)",
               [app_id_int, data])
        conn.commit()
    except Exception:
        continue

    cnt += 1
    if cnt % 100 == 0:
        print(cnt)

conn.close()

C:\Users\erichter\AppData\Local\Temp\ipykernel_25576\619492453.py:28: DeprecationWarning: LANCZOS is deprecated and will be removed in Pillow 10 (2023-07-01). Use Resampling.LANCZOS instead.
  img.thumbnail(size, Image.LANCZOS)


In [20]:
from PIL import Image
from io import BytesIO
import requests
from src.db.mysql import DbConnection

conn = DbConnection()
rows = conn.select_all("""
SELECT icon FROM scraped_data.apps_icons
""")

img = Image.open(BytesIO(rows[0]["icon"]))

conn.close()

img
img.save("test.png", format='PNG')

In [4]:
conn.close()

In [2]:
from src.server.models.scraper_model import ScraperModel
from bs4 import BeautifulSoup
from src.utils.gembase_utils import GembaseUtils
import json

from src.db.mysql import DbConnection

conn = DbConnection()

ScraperModel.scrap_app(
    conn=conn,
    app_id_in_store="com.king.candycrushsodasaga",
    store=1
)

conn.close()

In [4]:
conn.close()